# Зеркальный спуск (20 баллов)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from abc import ABCMeta, abstractmethod
from collections import OrderedDict
from scipy.special import softmax

## Основная часть (10 баллов)

__Задача 1.__ Задача многорукого бандита (Multi-Armed Bandit) — это классическая модель обучения с подкреплением при частичной наблюдаемости, где агент последовательно выбирает одно из $K$ действий ("рук") и получает случайную награду, зависящую только от выбранного действия. 

Формальная постановка:
- На каждом шаге $t=1, 2, \ldots, T$ агент выбирает действие $a_t \in \{1, \ldots, K \}$.
- Наблюдает награду $r \sim \mathcal{D}_{a_t}$, где $\mathcal{D}_i$ — неизвестное распределение награды для руки i.
- Цель — максимизировать суммарную ожидаемую награду или, что эквивалентно, минимизировать regret:

$$
R(T) = T \mu^* - \mathbb{E} \left[ \sum_{t = 1}^T r_t \right],~ \text{где}~ \mu^* = \max\limits_{i = \overline{1, K}} \mathbb{E}_{r \sim \mathcal{D}_i} [r].
$$

Основная сложность — дилемма исследования и эксплуатации: необходимо балансировать между использованием уже найденных хороших действий и исследованием других для уточнения их оценок.

Мы будем рассматривать `BernoulliBandit`. Действие приносит вознаграждение $r$ в размере 1 с фиксированной вероятностью $0 \leq \mu_i \leq 1$, которая неизвестна агенту. Цель агента — свести к минимуму regret:

$$
R(T) = T\mu^* - \sum_{t = 1}^T \mu_{a_t}, \text{где}~ \mu^* = \max\limits_{i = \overline{1, K}} \mu_i.
$$

In [ ]:
class BernoulliBandit():
    def __init__(self, n_actions=5):
        self._probs = np.random.random(n_actions)

    @property
    def action_count(self):
        return len(self._probs)

    def pull(self, action):
        if np.any(np.random.random() > self._probs[action]):
            return 0.0
        return 1.0

    def optimal_reward(self):
        return np.max(self._probs)

    def action_value(self, action):
        return self._probs[action]

Класс `AbstractAgent` необходим, чтобы от него наследоваться.

In [ ]:
class AbstractAgent(metaclass=ABCMeta):
    def init_actions(self, n_actions):
        self._successes = np.zeros(n_actions)
        self._failures = np.zeros(n_actions)
        self._total_pulls = 0

    def init_actions_agent(self, n_actions):
        pass
        
    @abstractmethod
    def get_action(self, action=None, reward=None):
        pass

    def update(self, action, reward):
        self._total_pulls += 1
        if reward == 1:
            self._successes[action] += 1
        else:
            self._failures[action] += 1

    @property
    def name(self):
        return self.__class__.__name__

__а) (1 балл)__ В данном пункте необходимо реализовать одну из самых простых стратегий. В ней агент с вероятностью $\varepsilon$ выбирает случайную руку, а с вероятностью $1 - \varepsilon$ берет ту руку, у которой максимальная средняя награда на данный момент. Логика проста — если положить $\varepsilon = 0$, то это будет жадный алгоритм, в котором выберется лучшая рука на данный момент, но эта стратегия не всегда будет оптимальной. Для этого вводится случайность — с некоторой маленькой вероятностью выбирается какая-то рука, для того, чтобы была возможность вылезти из локального минимума. 

**Псевдокод алгоритма**

---

_Инициализация:_

- Вероятность сэмплирования не жадной руки $\varepsilon \geq 0$

---

*$t$-ая итерация:*

Сэмплируем руку:
   - С вероятностью $\varepsilon$: $a_t \sim U \left(\overline{1, K}\right)$
   - С вероятностью $1 - \varepsilon$: $a_t = \arg \max\limits_{i = \overline{1, K}} \hat{\mu}_i$  

In [ ]:
class EpsilonGreedyAgent(AbstractAgent):
    def __init__(self, epsilon=0.01):
        self._epsilon = epsilon

    def get_action(self, action=None, reward=None):
        
        # YOUR CODE HERE

    @property
    def name(self):
        return self.__class__.__name__ + "(epsilon={})".format(self._epsilon)

Ниже представлена функция взаимодействия агента с бандитом.

In [ ]:
def get_regret(env, agents, n_steps=1000, n_trials=10):
    scores = OrderedDict({agent.name: [0.0 for step in range(n_steps)] for agent in agents})
    action = 0
    reward = 0

    for trial in range(n_trials):

        for a in agents:
            a.init_actions(env.action_count)
            a.init_actions_agent(env.action_count)

        for i in range(n_steps):
            optimal_reward = env.optimal_reward()

            for agent in agents:
                action = agent.get_action(action, reward)
                reward = env.pull(action)
                agent.update(action, reward)
                scores[agent.name][i] += optimal_reward - env.action_value(action)

    for agent in agents:
        scores[agent.name] = np.cumsum(scores[agent.name]) / n_trials

    return scores

In [ ]:
def plot_regret(agents, scores):
    for agent in agents:
        plt.plot(scores[agent.name])

    plt.legend([agent.name for agent in agents])

    plt.ylabel("Regret")
    plt.xlabel("Шаги")

    plt.show()

Постройте графики regret от номера шага для агентов с различным параметром `epsilon`.

In [ ]:
# Ваше решение (Code)

Какой `epsilon` получился оптимальным и почему?

In [ ]:
# Ваше решение (Markdown)

__б) (1 балл)__ Теперь рассмотрим чуть более сложную стратегию. Называется она Upper Confidence Bound (UCB) — из названия понятно, что нужно будет строить доверительные интервалы. Интуиция почти жадная — выбираем ту руку, для которой построенная оценка $\hat{\mu}_i$ имеет самую большую верхнюю оценку. 

**Псевдокод алгоритма**

---

_Инициализация:_

- Инициализируем параметр $C \geq 1$ — он отвечает за уверенность

---

_$t$-ая итерация:_

- Сэмплируем руку:
$$
a_t = \arg \max_{i = \overline{1, K}} \left( \hat{\mu}_i + C \sqrt{\frac{\log t}{n_i + 10^{-8}}} \right)
$$ 

Почему такая странная формула для выбора руки? В [теории](https://homes.di.unimi.it/~cesabian/Pubblicazioni/ml-02.pdf) указано, что при выборе $C = 1$ гарантируется, что рост регрета будет логарифмическим, что является оптимальной оценкой. 

In [ ]:
class UCBAgent(AbstractAgent):
    def __init__(self, C=1):
        self._C = C
    
    def get_action(self, action=None, reward=None):
        
        # YOUR CODE HERE
    
    @property
    def name(self):
        return self.__class__.__name__ + "(C={})".format(self._C)

Постройте графики regret от номера шага для `UCBAgent` агента с различным параметром `С`. Добавьте на график лучшего `EpsilonGreedyAgent` агента. 

In [ ]:
# Ваше решение (Code)

__в) (4 балла)__ Теперь поговорим про градиентные методы. При большом $K$ или при наличии корреляций между руками эффективны методы, основанные на онлайн-оптимизации на вероятностном симплексе:

$$
\Delta_{K - 1} = \left\{\pi \in \mathbb{R}^K \mid \pi \succeq 0,~ \boldsymbol{1}^\top \pi = 1 \right\}.
$$

Здесь $\pi_i$ интерпретируется как вероятность выбора $i$-ой руки. Политику $\pi$ можно параметризовать логитами $\theta \in \mathbb{R}^K$, а распределение над руками тогда

$$
\pi_i =  \frac{e^{\theta_i}}{\sum\limits_{a = 1}^K e^{\theta_{a}}}.
$$

Тогда можно рассмотреть минимизацию ожидаемого regret на шаге $t$:

$$
f_t(\theta) = 1 - \mathbb{E}_{a \sim \pi} [r_t(a)].
$$

Докажите, используя _log-derivative trick_, что градиент имеет вид 

$$
\nabla_{\theta} f_t(\theta) = -\mathbb{E}_{a \sim \pi} \left[r_t(a) (e_a - \pi) \right].
$$

In [ ]:
# Ваше решение (Markdown)

Сделаем градиентный спуск, где обновление происходит только по наблюдаемой руке:

$$
\theta_{a_t}^{t + 1} = \theta_{a_t}^t + \gamma_t \cdot r_t \cdot (1 - \pi_{a_t}).
$$

In [ ]:
class GDAgent(AbstractAgent):
    def __init__(self, gamma=0.1, n_actions=5):
        self._gamma = gamma
        self._tetas = np.zeros(n_actions)
        
    def init_actions_agent(self, n_actions):
        self._tetas = np.zeros(n_actions)
    
    def get_action(self, action, reward):
        
        # YOUR CODE HERE
    
    @property
    def name(self):
        return self.__class__.__name__ + "(gamma={})".format(self._gamma)

Постройте графики regret от номера шага для `GDAgent` агента с различным параметром `gamma`. Добавьте на график лучших `EpsilonGreedyAgent` и `UCBAgent` агентов. 

In [ ]:
# Ваше решение (Code)

__г) (4 балла)__ Так как мы работаем с вероятностными распределениями, то для зеркального спуска логично взять в качестве дивергенции Брэгмана расстояние Кульбака—Лейблера:

$$
KL(p \parallel q) = \sum_{i = 1}^K p_i \log \frac{p_i}{q_i}.
$$

Как и в случае градиентной политики, зеркальный спуск в контексте многорукого бандита также оперирует распределением вероятностей над действиями. Однако он оптимизирует не логиты $\theta$, а работает в пространстве вероятностей $\pi \in \Delta_{K - 1}$. Так же, как в прошлом задании, рассмотрим минимизацию ожидаемого regret на шаге $t$:

$$
f_t(\pi) = 1 - \mathbb{E}_{a \sim \pi} [r_t(a)].
$$

Алгоритм зеркального спуска на шаге $t$ решает следующую задачу:

$$
\pi^{t + 1} = \arg \min_{\pi \in \Delta_{K - 1}} \left\{ \langle \gamma_t g^t, \pi \rangle + KL(\pi \parallel \pi^t) \right\}.
$$

Покажите, что для $KL$ дивергенции решение этой задачи имеет вид:

$$
\pi_i^{t + 1} = \frac{\pi_i^t \cdot \exp(-\gamma_t g_i^t)}{\sum\limits_{j = 1}^K \pi_j^t \cdot \exp(-\gamma_t g_j^t)}.
$$

In [ ]:
# Ваше решение (Markdown)

Агент наблюдает только $r_t$ для выбранного действия $a_t$, а не весь вектор $g^t$. Чтобы получить несмещённую оценку градиента, используется _importance weighting_:
$$
\hat{g}_i^t = -\frac{\mathbb{I}\{i = a_t\} \cdot r_t}{\pi_i^t}.
$$

Подставляя эту оценку в правило обновления, получаем:

$$
\pi_i^{t + 1} = \frac{\pi_i^t \cdot \exp\left(\gamma_t \frac{\mathbb{I}\{i = a_t\} \cdot r_t}{\pi_i^t} \right)}{\sum\limits_{j = 1}^K \pi_j^t \cdot \exp\left(\gamma_t \frac{\mathbb{I}\{j = a_t\} \cdot r_t}{\pi_i^t} \right)}.
$$

In [ ]:
class MDAgent(AbstractAgent):
    def __init__(self, gamma=0.1, n_actions=5):
        self._gamma = gamma
        self._tetas = np.zeros(n_actions)

    def init_actions_agent(self, n_actions):
        self._tetas = np.zeros(n_actions)
        
    def get_action(self, action, reward):
        
        # YOUR CODE HERE
    
    @property
    def name(self):
        return self.__class__.__name__ + "(gamma={})".format(self._gamma)

Постройте графики regret от номера шага для `MDAgent` агента с различным параметром `gamma`. Добавьте на график лучших `EpsilonGreedyAgent`, `UCBAgent`, `GDAgent` агентов. 

In [ ]:
# Ваше решение (Code)

## Дополнительная часть (10 баллов)

In [ ]:
import os
import io
import glob
import base64
import random
from collections import deque

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from torch.nn.utils.convert_parameters import vector_to_parameters, parameters_to_vector

import gymnasium as gym
from IPython.display import clear_output, display, HTML

__Задача 2.__ В этой задаче вам будет предложено реализовать два практических метода обучения с подкреплением — TRPO и PPO, записать видео и посмотреть, как именно работает ваша модель.

__Литература:__

1. [Статья](http://proceedings.mlr.press/v37/schulman15.pdf) про Trust Region Policy Optimization;
2. [Статья](https://arxiv.org/abs/1707.06347.pdf) про Proximal Policy Optimization;
3. [Лекция](https://rail.eecs.berkeley.edu/deeprlcourse-fa17/f17docs/lecture_13_advanced_pg.pdf), объясняющая почти всю теорию по этим моделям.

__а) (2 балла)__ Мы будем использовать модель _Actor-to-Critic_ (_A2C_): модель _actor_ отвечает за параметрическую политику $\pi_\theta(a | s)$, а модель _critic_ — за функцию ценности $V_\phi(s)$. Градиент по политике с функцией advantage $A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s)$ имеет вид:

$$
\nabla_\theta J(\theta) = \mathbb{E}_{(s, a) \sim \mu_\pi} \left[\nabla_\theta \log \pi_\theta(a | s) A^\pi(s, a) \right].
$$

Advantage оценивается одношаговой разностью (_temporal difference_):

$$
A^k = r(s^k, a^k) + \gamma V_\phi(s^{k + 1}) - V_\phi(s^k).
$$

Функции потерь:

$$
\mathcal{L}_{\text{actor}}(\theta) = -\mathbb{E} \left[\log \pi_\theta(a^k | s^k) A^k \right], \qquad \mathcal{L}_{\text{critic}}(\phi) = \frac{1}{2} \mathbb{E} \left[(A^k)^2 \right].
$$

По _actor_ делается подъём, по _critic_ — спуск, что отсылает нас к седловым задачам и градиентному спуску—подъёму.

Построим модель A2C. Для этого вам нужно будет реализовать две нейронные сети:

1. _Actor_ имеет следующую архитектуру:
    - Линейный слой;
    - ReLU;
    - Выходной линейный слой, возвращающий логиты действий, по которым строится категориальное распределение политики;
    - Инициализация весов и смещений выходного слоя должна быть равномерным распределением (для этого дана функция `init_layer_uniform`).

В качестве среды мы будем использовать `CartPole-v1`. Это среда с дискретными действиями: на каждом шаге тележку нужно толкнуть влево или вправо так, чтобы шест на ней не упал.

In [ ]:
def init_layer_uniform(layer, init_w= 3e-3):
    """
    Инициализация параметров слоя равномерным распределением.

    Параметры:
        layer (nn.Module): Линейный слой, параметры которого необходимо инициализировать
        init_w (float): Граница равномерного распределения (параметры инициализируются из [-init_w, init_w])

    Возвращает:
        layer (nn.Module): Слой с инициализированными параметрами
    """
    layer.weight.data.uniform_(-init_w, init_w)
    layer.bias.data.uniform_(-init_w, init_w)
    return layer

In [ ]:
class Actor(nn.Module):
    """
    Реализация модели actor.

    Параметры:
        in_dim (int): Размерность пространства состояний
        out_dim (int): Число действий

    Возвращает:
        action (torch.Tensor): Сэмплированное действие
        dist (torch.distributions.Categorical): Категориальное распределение политики
    """
    def __init__(self, in_dim, out_dim):
        super(Actor, self).__init__()

        # YOUR CODE HERE

    def forward(self, state):
        """
        Параметры:
            state (torch.Tensor): Текущее состояние среды
        """

        # YOUR CODE HERE

        return action, dist

2. _Critic_ имеет следующую архитектуру:
    - Линейный слой;
    - ReLU;
    - Линейный слой;
    - Инициализация весов и смещений последнего слоя должна быть равномерным распределением (для этого дана функция `init_layer_uniform`).

In [ ]:
class Critic(nn.Module):
    """
    Реализация модели critic.

    Параметры:
        in_dim (int): Размерность пространства состояний

    Возвращает:
        value (torch.Tensor): Оценка функции ценности V(s)
    """
    def __init__(self, in_dim):
        super(Critic, self).__init__()

        # YOUR CODE HERE

    def forward(self, state):
        """
        Параметры:
            state (torch.Tensor): Текущее состояние среды
        """
        # YOUR CODE HERE

        return value

Для уменьшения дисперсии используется [`GAE`](https://arxiv.org/pdf/1506.02438) алгоритм. У него есть два гиперпараметра: $\tau \in [0, 1]$ и $\gamma \in [0, 1]$.

In [ ]:
def compute_gae(next_value, rewards, masks, values, gamma, tau):
    """
    Вычисление обобщённой оценки преимущества (GAE).

    Параметры:
        next_value (torch.Tensor): Оценка V для следующего состояния
        rewards (list): Список наград r_k
        masks (list): Список индикаторов продолжения эпизода
        values (list): Список оценок V
        gamma (float): Коэффициент дисконтирования
        tau (float): Параметр сглаживания

    Возвращает:
        returns (list): Список оценок return
    """
    values = values + [next_value]
    gae = 0
    returns = deque()

    for step in reversed(range(len(rewards))):
        delta = rewards[step] + gamma * values[step + 1] * masks[step] - values[step]
        gae = delta + gamma * tau * masks[step] * gae
        returns.appendleft(gae + values[step])

    return list(returns)

Общая часть агентов TRPO и PPO вынесена в класс `BaseAgent`. Агенты наследуются от него и отличаются только методом `update_model`.

| Метод | Описание |
|---|---|
| `select_action` | Выбор действия в текущем состоянии |
| `step` | Выполнение действия и получение ответа среды |
| `get_batch` | Подсчёт GAE и сборка накопленной памяти в тензоры |
| `update_model` | Обновление моделей actor и critic |
| `train` | Обучение агента в течение `num_frames` шагов |
| `test` | Тестирование агента (1 эпизод) с записью видео |
| `_plot` | Визуализация процесса обучения |

In [ ]:
class BaseAgent:
    """
    Общая часть агентов: сбор переходов, обучение, тестирование и визуализация.

    Параметры:
        env (gym.Env): Среда обучения
        gamma (float): Коэффициент дисконтирования
        tau (float): Параметр GAE для сглаживания advantage
        rollout_len (int): Число шагов в среде между обновлениями
        seed (int): Seed для воспроизводимости
    """
    def __init__(self, env, gamma, tau, rollout_len, seed):
        self.env = env
        self.gamma = gamma
        self.tau = tau
        self.rollout_len = rollout_len
        self.seed = seed

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        obs_dim = env.observation_space.shape[0]
        action_dim = env.action_space.n

        self.actor = Actor(obs_dim, action_dim).to(self.device)
        self.critic = Critic(obs_dim).to(self.device)

        self.is_test = False
        self.clear_memory()

    def clear_memory(self):
        """
        Очистка памяти переходов.
        """
        self.states, self.actions, self.rewards = [], [], []
        self.values, self.masks, self.log_probs = [], [], []

    def select_action(self, state):
        """
        Выбор действия по текущей политике.

        Параметры:
            state (np.ndarray): Текущее состояние среды

        Возвращает:
            action (int): Выбранное действие
        """
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        action, dist = self.actor(state)

        if self.is_test:
            action = dist.probs.argmax(dim=-1)
        else:
            self.states.append(state)
            self.actions.append(action)
            self.values.append(self.critic(state))
            self.log_probs.append(dist.log_prob(action))

        return action.item()

    def step(self, action):
        """
        Выполнение шага в среде.

        Параметры:
            action (int): Действие агента

        Возвращает:
            next_state (np.ndarray): Следующее состояние
            reward (float): Полученная награда
            done (bool): Флаг завершения эпизода
        """
        next_state, reward, terminated, truncated, _ = self.env.step(action)
        done = terminated or truncated

        if not self.is_test:
            self.rewards.append(torch.FloatTensor([[reward]]).to(self.device))
            self.masks.append(torch.FloatTensor([[1 - done]]).to(self.device))

        return next_state, reward, done

    def get_batch(self, next_state):
        """
        Подсчёт GAE и сборка накопленной памяти в тензоры.

        Параметры:
            next_state (np.ndarray): Состояние после последнего шага

        Возвращает:
            states (torch.Tensor): Состояния
            actions (torch.Tensor): Действия
            returns (torch.Tensor): Оценки return
            log_probs (torch.Tensor): Лог-вероятности действий при старой политике
            advantages (torch.Tensor): Оценки advantage
        """
        next_state = torch.FloatTensor(next_state).unsqueeze(0).to(self.device)
        next_value = self.critic(next_state)
        returns = compute_gae(next_value, self.rewards, self.masks, self.values, self.gamma, self.tau)

        states = torch.cat(self.states)
        actions = torch.cat(self.actions).detach()
        returns = torch.cat(returns).detach()
        values = torch.cat(self.values).detach()
        log_probs = torch.cat(self.log_probs).detach()

        self.clear_memory()
        return states, actions, returns, log_probs, (returns - values).squeeze(-1)

    def update_model(self, next_state):
        """
        Обновление параметров actor и critic (реализуется в наследниках).
        """
        raise NotImplementedError

    def train(self, num_frames):
        """
        Обучение агента.

        Параметры:
            num_frames (int): Общее число шагов взаимодействия со средой
        """
        self.is_test = False
        state, _ = self.env.reset(seed=self.seed)

        actor_losses, critic_losses = [], []
        scores = []
        score = 0

        for frame_idx in range(1, num_frames + 1):
            action = self.select_action(state)
            next_state, reward, done = self.step(action)

            state = next_state
            score += reward

            if done:
                state, _ = self.env.reset(seed=self.seed)
                scores.append(score)
                score = 0

            if frame_idx % self.rollout_len == 0:
                actor_loss, critic_loss = self.update_model(next_state)
                actor_losses.append(actor_loss)
                critic_losses.append(critic_loss)
                self._plot(frame_idx, scores, actor_losses, critic_losses)

        self.env.close()

    def test(self, video_folder):
        """
        Тестирование обученного агента с записью видео.

        Параметры:
            video_folder (str): Папка для сохранения видео
        """
        self.is_test = True

        tmp_env = self.env
        self.env = gym.wrappers.RecordVideo(self.env, video_folder=video_folder)

        state, _ = self.env.reset(seed=self.seed)
        done = False
        score = 0

        while not done:
            action = self.select_action(state)
            state, reward, done = self.step(action)
            score += reward

        print("score: ", score)
        self.env.close()

        self.env = tmp_env

    def _plot(self, frame_idx, scores, actor_losses, critic_losses):
        """
        Визуализация процесса обучения.

        Параметры:
            frame_idx (int): Текущий шаг обучения
            scores (list): История reward
            actor_losses (list): История потерь актора
            critic_losses (list): История потерь критика
        """
        subplot_params = [
            (131, f"frame {frame_idx}. score: {np.mean(scores[-10:]) if scores else 0:.1f}", scores),
            (132, "actor_loss", actor_losses),
            (133, "critic_loss", critic_losses),
        ]

        clear_output(True)
        plt.figure(figsize=(30, 5))
        for loc, title, values in subplot_params:
            plt.subplot(loc)
            plt.title(title)
            plt.plot(values)
        plt.show()

Создадим среду `CartPole-v1`. Можно также посмотреть [код](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/classic_control/cartpole.py) этой среды в `gymnasium`.

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

В обучении с подкреплением результат очень сильно зависит от случайности, поэтому заранее зафиксируем seed.

In [ ]:
def seed_torch(seed):
    torch.manual_seed(seed)
    if torch.backends.cudnn.enabled:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


seed = 777
random.seed(seed)
np.random.seed(seed)
seed_torch(seed)

Ниже даны функции визуализации: достаточно передать в них обученного агента, у которого реализован метод `test`.

In [ ]:
def ipython_show_video(path):
    """
    Отображение видео в ноутбуке.

    Параметры:
        path (str): Путь к видеофайлу
    """
    if not os.path.isfile(path):
        raise NameError("Cannot access: {}".format(path))

    video = io.open(path, "r+b").read()
    encoded = base64.b64encode(video)

    display(
        HTML(
            data="""
        <video width="320" height="240" alt="test" controls>
        <source src="data:video/mp4;base64,{0}" type="video/mp4"/>
        </video>
        """.format(encoded.decode("ascii"))
        )
    )


def show_latest_video(video_folder):
    """
    Отображение последнего записанного видео из папки.

    Параметры:
        video_folder (str): Папка с видео

    Возвращает:
        latest_file (str): Путь к отображённому видео
    """
    list_of_files = glob.glob(os.path.join(video_folder, "*.mp4"))
    latest_file = max(list_of_files, key=os.path.getctime)
    ipython_show_video(latest_file)
    return latest_file

__б) (4 балла)__ Перейдём к методу Trust Region Policy Optimization (TRPO).

__Вывод метода TRPO__

С помощью отбора по важности (_importance sampling_) относительно старой политики $\pi_{\theta^{\text{old}}}$ ожидаемую награду новой политики можно приблизить суррогатной целевой функцией:

$$
\mathcal{L}(\theta) = \mathbb{E}_{s, a \sim \pi_{\theta^{\text{old}}}} \left[\frac{\pi_{\theta}(a | s)}{\pi_{\theta^{\text{old}}}(a | s)} A^{\pi_{\theta^{\text{old}}}}(s, a) \right].
$$

Это приближение хорошее, только пока новая политика близка к старой, поэтому TRPO решает задачу с ограничением на среднюю дивергенцию Кульбака—Лейблера:

$$
\begin{align*}
\max_{\theta} ~~&~~ \mathcal{L}(\theta) \\
\text{s.t.} ~~&~~ \mathbb{E}_{s \sim d^{\pi_{\theta^{\text{old}}}}} \left[D_{\text{KL}}(\pi_{\theta^{\text{old}}}(\cdot | s) \| \pi_{\theta}(\cdot | s)) \right] \leq \delta.
\end{align*}
$$

Здесь $\delta$ — гиперпараметр, контролирующий, насколько сильно может измениться политика за один шаг. Разложим целевую функцию до первого порядка, а ограничение — до второго порядка в окрестности $\theta^{\text{old}}$:

$$
\mathcal{L}(\theta) \approx \mathcal{L}(\theta^{\text{old}}) + g^\top (\theta - \theta^{\text{old}}), \qquad \mathbb{E}_{s} \left[D_{\text{KL}} \right] \approx \frac{1}{2} (\theta - \theta^{\text{old}})^\top F (\theta - \theta^{\text{old}}),
$$

где $g = \nabla_{\theta} \mathcal{L}(\theta) \big|_{\theta = \theta^{\text{old}}}$, а $F$ — матрица Фишера:

$$
F = \mathbb{E}_{s, a \sim \pi_{\theta^{\text{old}}}} \left[\nabla_{\theta} \log \pi_{\theta}(a | s) \nabla_{\theta} \log \pi_{\theta}(a | s)^\top \right] \Big|_{\theta = \theta^{\text{old}}}.
$$

Используя метод множителей Лагранжа, получаем явную формулу шага:

$$
\theta^{\text{new}} = \theta^{\text{old}} + \sqrt{\frac{2 \delta}{g^\top F^{-1} g}} F^{-1} g.
$$

Вычислять $F^{-1}$ напрямую затратно, поэтому TRPO находит $F^{-1} g$ методом сопряжённых градиентов, которому нужны только произведения $F v$.

Теперь, когда мы разобрались, как устроен TRPO, допишите класс `TRPOAgent`.

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальные параметры политики $\theta^0 \in \mathbb{R}^d$, где $d$ — число параметров модели _actor_
- Параметр ограничения на дивергенцию Кульбака—Лейблера $\delta > 0$

---

$k$_-ая итерация:_

1. Собрать траектории, выполняя политику $\pi_{\theta^k}(a | s)$.

2. Вычислить оценки advantage $A^k$ по собранной траектории с помощью обобщённой оценки преимущества (GAE):

$$
A^k = \delta^k + \gamma \tau A^{k + 1}, \quad \delta^k = r^k + \gamma V(s^{k + 1}) - V(s^k).
$$

3. Вычислить градиент политики:

$$
g = \mathbb{E}_{s, a \sim \pi_{\theta^k}} \left[\nabla_{\theta} \log \pi_{\theta}(a | s) \big|_{\theta = \theta^k} A^{\pi_{\theta^k}}(s, a) \right].
$$

4. Оценить матрицу Фишера:

$$
F(\theta^k) = \mathbb{E}_{s, a \sim \pi_{\theta^k}} \left[\nabla_{\theta} \log \pi_{\theta}(a | s) \nabla_{\theta} \log \pi_{\theta}(a | s)^\top \right] \Big|_{\theta = \theta^k}.
$$

5. Вычислить направление обновления политики, решив методом сопряжённых градиентов систему:

$$
F(\theta^k) x = g.
$$

6. Обновить параметры политики:

$$
\theta^{k + 1} = \theta^k + \sqrt{\frac{2 \delta}{g^\top x}} x.
$$

После этого обучите агента. Значения гиперпараметров в ячейке ниже не являются финальными, а нужны лишь как отправная точка: подберите гиперпараметры и выпишите лучшую конфигурацию.

In [ ]:
class TRPOAgent(BaseAgent):
    """
    Реализация агента Trust Region Policy Optimization.

    Параметры:
        env (gym.Env): Среда обучения
        gamma (float): Коэффициент дисконтирования
        tau (float): Параметр GAE для сглаживания advantage
        delta (float): Ограничение на KL-дивергенцию при обновлении политики
        cg_iters (int): Число итераций метода сопряжённых градиентов
        cg_damping (float): Коэффициент демпфирования в произведении матрицы Фишера на вектор
        backtrack_coeff (float): Коэффициент уменьшения шага при поиске шага
        backtrack_iters (int): Максимальное число итераций поиска шага
        value_lr (float): Скорость обучения критика
        value_iters (int): Число шагов обучения критика за одно обновление
        rollout_len (int): Число шагов в среде между обновлениями
        seed (int): Seed для воспроизводимости
    """
    def __init__(self, env, gamma=0.99, tau=0.95, delta=0.01, cg_iters=10, cg_damping=0.1, backtrack_coeff=0.5,
                 backtrack_iters=10, value_lr=1e-3, value_iters=20, rollout_len=1000, seed=777):
        super().__init__(env, gamma, tau, rollout_len, seed)

        self.delta = delta
        self.cg_iters = cg_iters
        self.cg_damping = cg_damping
        self.backtrack_coeff = backtrack_coeff
        self.backtrack_iters = backtrack_iters
        self.value_iters = value_iters
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=value_lr)

    def surrogate(self, states, actions, advantages, old_log_probs):
        """
        Вычисление суррогатной целевой функции.

        Параметры:
            states (torch.Tensor): Состояния
            actions (torch.Tensor): Действия
            advantages (torch.Tensor): Оценки advantage
            old_log_probs (torch.Tensor): Лог-вероятности действий при старой политике

        Возвращает:
            surr (torch.Tensor): Значение суррогатной функции
        """
        # YOUR CODE HERE

        return surr

    def kl(self, states, old_dist):
        """
        Средняя KL-дивергенция между старой и текущей политиками.

        Параметры:
            states (torch.Tensor): Состояния
            old_dist (torch.distributions.Categorical): Распределение старой политики

        Возвращает:
            kl (torch.Tensor): Значение KL-дивергенции
        """
        _, dist = self.actor(states)
        return torch.distributions.kl_divergence(old_dist, dist).mean()

    def fisher_vector_product(self, states, old_dist, p):
        """
        Произведение матрицы Фишера на вектор (через гессиан KL-дивергенции).

        Параметры:
            states (torch.Tensor): Состояния
            old_dist (torch.distributions.Categorical): Распределение старой политики
            p (torch.Tensor): Вектор

        Возвращает:
            fvp (torch.Tensor): Произведение F p
        """
        kl = self.kl(states, old_dist)
        grads = torch.autograd.grad(kl, self.actor.parameters(), create_graph=True)
        grad_kl = torch.cat([g.view(-1) for g in grads])
        grads = torch.autograd.grad(grad_kl @ p, self.actor.parameters())
        return torch.cat([g.reshape(-1) for g in grads]) + self.cg_damping * p

    def conjugate_gradient(self, states, old_dist, b):
        """
        Решение системы F x = b методом сопряжённых градиентов.

        Параметры:
            states (torch.Tensor): Состояния
            old_dist (torch.distributions.Categorical): Распределение старой политики
            b (torch.Tensor): Правая часть

        Возвращает:
            x (torch.Tensor): Приближённое решение
        """
        # YOUR CODE HERE

        return x

    def update_model(self, next_state):
        """
        Обновление параметров actor и critic.

        Параметры:
            next_state (np.ndarray): Состояние после последнего шага

        Возвращает:
            actor_loss (float): Значение функции потерь актора
            critic_loss (float): Значение функции потерь критика
        """
        states, actions, returns, old_log_probs, advantages = self.get_batch(next_state)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # 1. Обновление critic
        for _ in range(self.value_iters):
            critic_loss = F.mse_loss(self.critic(states), returns)
            self.critic_optimizer.zero_grad()
            critic_loss.backward()
            self.critic_optimizer.step()

        # 2. Распределение старой политики
        with torch.no_grad():
            _, old_dist = self.actor(states)

        # 3-5. Вычислите градиент g суррогатной функции surr, решите систему F x = g
        # методом сопряжённых градиентов и найдите полный шаг full_step

        # YOUR CODE HERE

        # 6. Поиск шага: уменьшаем шаг, пока не выполнено ограничение на KL и не выросла суррогатная функция
        if not torch.isfinite(full_step).all():
            return -surr.item(), critic_loss.item()

        old_params = parameters_to_vector(self.actor.parameters())
        for i in range(self.backtrack_iters):
            vector_to_parameters(old_params + self.backtrack_coeff ** i * full_step, self.actor.parameters())
            try:
                with torch.no_grad():
                    kl = self.kl(states, old_dist)
                    new_surr = self.surrogate(states, actions, advantages, old_log_probs)
            except ValueError:
                # шаг слишком большой: политика выдаёт некорректные значения
                continue
            if kl <= self.delta and new_surr >= surr:
                break
        else:
            vector_to_parameters(old_params, self.actor.parameters())

        return -surr.item(), critic_loss.item()

In [ ]:
num_frames = 100000

trpo_agent = TRPOAgent(
    env,
    gamma=0.99,
    tau=0.95,
    delta=0.01,
    cg_iters=10,
    cg_damping=1e-1,
    backtrack_coeff=0.5,
    backtrack_iters=10,
    value_lr=1e-3,
    value_iters=20,
    rollout_len=1000,
    seed=seed,
)

trpo_agent.train(num_frames)

In [ ]:
video_folder_trpo = "videos/trpo"

trpo_agent.test(video_folder=video_folder_trpo)

trpo = show_latest_video(video_folder=video_folder_trpo)
print("Played TRPO:", trpo)

__в) (4 балла)__ Перейдём к методу Proximal Policy Optimization (PPO).

__Вывод метода PPO__

TRPO вычислительно затратен из-за необходимости явно учитывать ограничение. PPO заменяет ограничение штрафом:

$$
L^{\text{PPO}}(\theta) = \mathbb{E}_{s, a \sim \pi_{\theta^{\text{old}}}} \left[r(\theta) A^{\pi_{\theta^{\text{old}}}}(s, a) - \beta D_{\text{KL}}(\pi_{\theta^{\text{old}}}(\cdot | s) \| \pi_{\theta}(\cdot | s)) \right], \qquad r(\theta) = \frac{\pi_{\theta}(a | s)}{\pi_{\theta^{\text{old}}}(a | s)}.
$$

Подобрать коэффициент $\beta$ затруднительно, поэтому PPO использует _усечённую_ (clipped) суррогатную функцию:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_{s, a \sim \pi_{\theta^{\text{old}}}} \left[\min \left(r(\theta) A^{\pi_{\theta^{\text{old}}}}(s, a),~ \text{clip}(r(\theta), 1 - \varepsilon, 1 + \varepsilon) A^{\pi_{\theta^{\text{old}}}}(s, a) \right) \right].
$$

Усечение работает как неявный штраф:

- если $A > 0$, то при $r(\theta) > 1 + \varepsilon$ дальнейшее увеличение вероятности действия не увеличивает целевую функцию;
- если $A < 0$, то при $r(\theta) < 1 - \varepsilon$ дальнейшее уменьшение вероятности действия не увеличивает целевую функцию.

Таким образом, обновления политики остаются осторожными и стабильными.

PPO несколько раз (`epoch` эпох) обновляет модель по накопленной памяти переходов. Функция `ppo_iter` выдаёт случайные мини-батчи из этой памяти.

In [ ]:
def ppo_iter(epoch, mini_batch_size, *tensors):
    """
    Генератор случайных мини-батчей из накопленной памяти.

    Параметры:
        epoch (int): Число эпох обновления
        mini_batch_size (int): Размер мини-батча
        tensors (torch.Tensor): Тензоры памяти одинаковой длины

    Возвращает:
        batch (list): Мини-батч из каждого тензора
    """
    batch_size = tensors[0].size(0)
    for _ in range(epoch):
        for _ in range(batch_size // mini_batch_size):
            ids = np.random.choice(batch_size, mini_batch_size)
            yield [t[ids] for t in tensors]

Допишите класс `PPOAgent`, который имеет ту же структуру, что и `TRPOAgent`.

**Псевдокод алгоритма**

---

_Инициализация:_

- Начальные параметры политики $\theta^0 \in \mathbb{R}^d$, где $d$ — число параметров модели _actor_
- Параметр ограничения на дивергенцию Кульбака—Лейблера $\delta > 0$

---

$k$_-ая итерация:_

1. Собрать траектории, используя политику $\pi_{\theta^k}(a | s)$.

2. Оценить advantage $A^{\pi_{\theta^k}}(s, a)$.

3. Найти новые параметры политики, приближённо максимизировав усечённую целевую функцию PPO несколькими эпохами градиентного подъёма, начиная с $\theta^k$:

$$
\theta^{k + 1} \approx \arg\max_{\theta} L^{\text{CLIP}}(\theta),
$$

где

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_{s, a \sim \pi_{\theta^k}} \left[\min \left(r(\theta) A^{\pi_{\theta^k}}(s, a),~ \text{clip}(r(\theta), 1 - \varepsilon, 1 + \varepsilon) A^{\pi_{\theta^k}}(s, a) \right) \right],
$$

$$
r(\theta) = \frac{\pi_{\theta}(a | s)}{\pi_{\theta^k}(a | s)}.
$$

После этого обучите агента. Значения гиперпараметров в ячейке ниже не являются финальными, а нужны лишь как отправная точка: подберите гиперпараметры и выпишите лучшую конфигурацию.

In [ ]:
class PPOAgent(BaseAgent):
    """
    Реализация агента Proximal Policy Optimization.

    Параметры:
        env (gym.Env): Среда обучения
        batch_size (int): Размер мини-батча
        gamma (float): Коэффициент дисконтирования
        tau (float): Параметр GAE для сглаживания advantage
        epsilon (float): Параметр усечения суррогатной функции
        epoch (int): Число эпох обновления на каждой итерации
        rollout_len (int): Число шагов в среде между обновлениями
        entropy_weight (float): Вес энтропийного слагаемого в функции потерь
        seed (int): Seed для воспроизводимости
    """
    def __init__(self, env, batch_size, gamma, tau, epsilon, epoch, rollout_len, entropy_weight, seed=777):
        super().__init__(env, gamma, tau, rollout_len, seed)
        self.batch_size = batch_size
        self.epsilon = epsilon
        self.epoch = epoch
        self.entropy_weight = entropy_weight

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=0.001)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=0.005)

    def update_model(self, next_state):
        """
        Обновление параметров actor и critic.

        Параметры:
            next_state (np.ndarray): Состояние после последнего шага

        Возвращает:
            actor_loss (float): Среднее значение функции потерь актора
            critic_loss (float): Среднее значение функции потерь критика
        """
        states, actions, returns, old_log_probs, advantages = self.get_batch(next_state)
        actor_losses, critic_losses = [], []

        for state, action, old_log_prob, return_, adv in ppo_iter(
            self.epoch, self.batch_size, states, actions, old_log_probs, returns, advantages
        ):
            # Вычислите actor_loss (усечённая суррогатная функция с энтропийным бонусом
            # с весом entropy_weight) и critic_loss (MSE между return_ и оценкой критика)

            # YOUR CODE HERE

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            self.critic_optimizer.zero_grad()
            critic_loss.backward()
            self.critic_optimizer.step()

            actor_losses.append(actor_loss.item())
            critic_losses.append(critic_loss.item())

        return np.mean(actor_losses), np.mean(critic_losses)

In [ ]:
num_frames = 100000

ppo_agent = PPOAgent(
    env,
    gamma=0.99,
    tau=0.95,
    batch_size=64,
    epsilon=0.2,
    epoch=10,
    rollout_len=2048,
    entropy_weight=0.01,
    seed=seed,
)

In [ ]:
ppo_agent.train(num_frames)

In [ ]:
video_folder_ppo = "videos/ppo"

ppo_agent.test(video_folder=video_folder_ppo)

ppo = show_latest_video(video_folder=video_folder_ppo)
print("Played PPO:", ppo)